In [110]:
library(tidyr)
library(dplyr)
library(ggplot2)
library(patchwork)
library(data.table)

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/module_projection_fxns.R")

setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

options(repr.plot.width=20, repr.plot.height=8, repr.plot.res=150)

Here I visualize module genes after "cleaning" the bulk data

In [41]:
mod_def <- "TopModPosBC"

### Bulk data and enrichments

In [ ]:
# # *** Bulk gene expression should be the same dataset used in to find modules and enrichments ***

# bulk_data_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed"
# bulk_expr_file <- "GTEx_cortex_counts_TMMF_SampleNetworks/All_02-25-12/GTEx_cortex_counts_TMMF_All_501_outliers_removed.csv"
# bulk_expr <- fread(bulk_expr_file, data.table=FALSE)
# colnames(bulk_expr)[1] <- "Gene"

# enrichment_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_Claude_marker_genes_enrichments_top_Qval_mods"
# # enrichment_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_MO_1271sets_enrichments_top_Qval_mods"

# top_mods_df <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)

# max_qval <- .05

# top_mods_df <- top_mods_df[!is.na(top_mods_df$Qval),]
# top_mods_df_subset <- top_mods_df[top_mods_df$Qval < max_qval,]

# dim(top_mods_df_subset)

[1] 56  7

In [4]:
# # *** Bulk gene expression should be the same dataset used in to find modules and enrichments ***

# bulk_data_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_All_370_outliers_removed"
# bulk_expr_file <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_SampleNetworks/All_02-59-24/GTEx_cortex_counts_TMMF_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_All_370_outliers_removed.csv"
# bulk_expr <- fread(bulk_expr_file, data.table=FALSE)
# colnames(bulk_expr)[1] <- "Gene"

# enrichment_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_All_370_outliers_removed_mergeParam0.95_subsetCutoff11.245_Modules_MO_1271sets_enrichments_top_Qval_mods"

# top_mods_df <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)

# max_qval <- 1e-20

# top_mods_df <- top_mods_df[!is.na(top_mods_df$Qval),]
# top_mods_df_subset <- top_mods_df[top_mods_df$Qval < max_qval,]

# dim(top_mods_df_subset)

In [ ]:
# *** Bulk gene expression should be the same dataset used in to find modules and enrichments ***

bulk_data_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned"
bulk_expr_file <- "data/cleaned/TopModPosBC/GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned.csv"
bulk_expr <- fread(bulk_expr_file, data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"

enrichment_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules_Claude_marker_genes_enrichments_top_Qval_mods"
# enrichment_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules_MO_1117sets_enrichments_top_Qval_mods"

top_mods_df <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)

max_qval <- .05

top_mods_df <- top_mods_df[!is.na(top_mods_df$Qval),]
top_mods_df_subset <- top_mods_df[top_mods_df$Qval < max_qval,]

dim(top_mods_df_subset)

[1] 65  7

### Prep single cell data

In [ ]:
library(anndata)

sc_data_source <- "ma_2022_counts_normalized"

adata <- anndata::read_h5ad("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/ma_2022/human/9198437d-f781-4215-bf4f-32c3177e57df.h5ad")

sc_expr <- t(as.matrix(adata$X))
sampleinfo <- adata$obs

total_expr <- colSums(sc_expr)
sc_expr_norm <- sweep(sc_expr, MARGIN=2, FUN="/", STATS=total_expr) * 1e4
sc_expr_norm <- data.frame(Gene=adata$var$feature_name, sc_expr_norm)

ctype_assignment_vec <- sampleinfo$subclass

In [ ]:
# library(anndata)

# sc_data_source <- "jorstad_2023_SMART-seq_counts_normalized"

# adata <- anndata::read_h5ad("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/jorstad_2023/SMART-seq/dfba29ea-d368-44c2-bb35-69d3e8f730ce.h5ad")

# sc_expr <- t(as.matrix(adata$X))
# sampleinfo <- adata$obs

# total_expr <- colSums(sc_expr)
# sc_expr_norm <- sweep(sc_expr, MARGIN=2, FUN="/", STATS=total_expr) * 1e4
# sc_expr_norm <- data.frame(Gene=adata$var$feature_name, sc_expr_norm)

# ctype_assignment_vec <- sampleinfo$subclass

## Visualize module genes

In [118]:
outdir <- paste0("figures/module_genes/", sc_data_source, "/", mod_def)

if (!dir.exists(outdir)) {
    dir.create(outdir, recursive=TRUE)
}

filename <- paste0(outdir, "/", enrichment_source, ".pdf")

max_genes <- 15

pdf(file=filename, width=10, height=8)

for (i in 1:nrow(top_mods_df_subset)) {
    print(paste("Module", i))

    mod <- top_mods_df_subset$Module[i]
    kME_path <- top_mods_df_subset$kME_path[i]
    mod_genes <- get_mod_genes(kME_path, mod, mod_def)

    plot_title <- paste(
        top_mods_df_subset$Cell_type[i], mod_def, 
        "\n", mod, top_mods_df_subset$Network[i]
    ) 
    plot_sub <- paste("Qval:", round(top_mods_df_subset$Qval[i], 4))

    # Plot gene expression over bulk samples
    
    mod_genes_subset <- na.omit(mod_genes[1:max_genes])
    plot_gene_expr_over_samples(bulk_expr, mod_genes_subset, plot_title, plot_sub, target_species=NULL)

    # Plot module genes in single cell data

    plot_gene_projections(sc_expr_norm, mod_genes, ctype_assignment_vec, plot_title, plot_sub, target_species=NULL) # ="mouse")
}

dev.off()

[1] "Module 1"
[1] "Module 2"
[1] "Module 3"
[1] "Module 4"
[1] "Module 5"
[1] "Module 6"
[1] "Module 7"
[1] "Module 8"
[1] "Module 9"
[1] "Module 10"
[1] "Module 11"
[1] "Module 12"
[1] "Module 13"
[1] "Module 14"
[1] "Module 15"
[1] "Module 16"
[1] "Module 17"
[1] "Module 18"
[1] "Module 19"
[1] "Module 20"
[1] "Module 21"
[1] "Module 22"
[1] "Module 23"
[1] "Module 24"
[1] "Module 25"
[1] "Module 26"
[1] "Module 27"
[1] "Module 28"
[1] "Module 29"
[1] "Module 30"
[1] "Module 31"
[1] "Module 32"
[1] "Module 33"
[1] "Module 34"
[1] "Module 35"
[1] "Module 36"
[1] "Module 37"
[1] "Module 38"
[1] "Module 39"
[1] "Module 40"
[1] "Module 41"
[1] "Module 42"
[1] "Module 43"
[1] "Module 44"
[1] "Module 45"
[1] "Module 46"
[1] "Module 47"
[1] "Module 48"
[1] "Module 49"
[1] "Module 50"
[1] "Module 51"
[1] "Module 52"
[1] "Module 53"
[1] "Module 54"
[1] "Module 55"
[1] "Module 56"
[1] "Module 57"
[1] "Module 58"
[1] "Module 59"
[1] "Module 60"
[1] "Module 61"
[1] "Module 62"
[1] "Module 63"
[

agg_record_251195938 
                   2

In [ ]:
# dev.off()

pdf 
  3